# Phase 0: Forex EDA (v2, split-aware)
Dataset: `daily_forex_rates.csv` (EUR base, daily rates, 2004-present).

**Leakage rule used throughout:** fixed rules (drop weekends, flag glitches) run before the split; anything *learned from data* (eligibility stats, clip bounds) is fitted on the **train window only** and applied to test.

## 1. Load and first look

In [ ]:
import polars as pl
import duckdb
import plotly.express as px
from datetime import timedelta

DATA = "../data/raw/daily_forex_rates.csv"
df = pl.read_csv(DATA, try_parse_dates=True)

print(df.shape)
print(df.schema)
df.head()

(505223, 5)
Schema({'currency': String, 'base_currency': String, 'currency_name': String, 'exchange_rate': Float64, 'date': Date})


currency,base_currency,currency_name,exchange_rate,date
str,str,str,f64,date
"""MXV""","""EUR""",null,2.287876,2026-10-07
"""GIP""","""EUR""","""Gibraltar Pound""",0.849793,2026-10-07
"""HTG""","""EUR""","""Haitian Gourde""",147.002605,2026-10-07
"""HRK""","""EUR""","""Croatian Kuna""",7.535912,2026-10-07
"""HNL""","""EUR""","""Honduran Lempira""",30.162333,2026-10-07


## 2. Quality checks

In [ ]:
print(df.null_count())
print("duplicates on (currency, date):", df.height - df.select(["currency", "date"]).n_unique())
print("rates <= 0:", df.filter(pl.col("exchange_rate") <= 0).height)
print("currency codes:", df["currency"].n_unique(),
      "| distinct names (non-null):", df["currency_name"].drop_nulls().n_unique())
print("base currency:", df["base_currency"].unique().to_list())
print("date range:", df["date"].min(), "->", df["date"].max())
df["exchange_rate"].describe()

shape: (1, 5)
┌──────────┬───────────────┬───────────────┬───────────────┬──────┐
│ currency ┆ base_currency ┆ currency_name ┆ exchange_rate ┆ date │
│ ---      ┆ ---           ┆ ---           ┆ ---           ┆ ---  │
│ u32      ┆ u32           ┆ u32           ┆ u32           ┆ u32  │
╞══════════╪═══════════════╪═══════════════╪═══════════════╪══════╡
│ 0        ┆ 0             ┆ 2899          ┆ 0             ┆ 0    │
└──────────┴───────────────┴───────────────┴───────────────┴──────┘
duplicates on (currency, date): 0
rates <= 0: 0
currency codes: 176 | distinct names (non-null): 169
base currency: ['EUR']
date range: 2004-08-30 -> 2026-10-07


statistic,value
str,f64
"""count""",505223.0
"""null_count""",0.0
"""mean""",6712.475508
"""std""",132491.944882
"""min""",0.000009
"""25%""",2.7781
"""50%""",18.827
"""75%""",181.268
"""max""",4.881841e6


## 3. Calendar and weekend policy
Weekend rows are ~0% until 2023, then jump. We keep trading days only so 'lag 1' always means 'previous trading day'. This is a fixed rule, so it is safe before the split.

In [ ]:
cal = (
    df.with_columns(
        year=pl.col("date").dt.year(),
        weekend=pl.col("date").dt.weekday() >= 6,   # Mon=1 ... Sun=7
    )
    .group_by("year")
    .agg(
        rows=pl.len(),
        currencies=pl.col("currency").n_unique(),
        weekend_share=pl.col("weekend").mean().round(3),
    )
    .sort("year")
)
print(cal)

df = df.filter(pl.col("date").dt.weekday() <= 5).sort(["currency", "date"])
print("after weekend filter:", df.shape)

shape: (23, 4)
┌──────┬───────┬────────────┬───────────────┐
│ year ┆ rows  ┆ currencies ┆ weekend_share │
│ ---  ┆ ---   ┆ ---        ┆ ---           │
│ i32  ┆ u32   ┆ u32        ┆ f64           │
╞══════╪═══════╪════════════╪═══════════════╡
│ 2004 ┆ 510   ┆ 6          ┆ 0.0           │
│ 2005 ┆ 1560  ┆ 6          ┆ 0.0           │
│ 2006 ┆ 1672  ┆ 7          ┆ 0.0           │
│ 2007 ┆ 1827  ┆ 7          ┆ 0.0           │
│ 2008 ┆ 1999  ┆ 9          ┆ 0.0           │
│ …    ┆ …     ┆ …          ┆ …             │
│ 2022 ┆ 33154 ┆ 132        ┆ 0.006         │
│ 2023 ┆ 37984 ┆ 170        ┆ 0.085         │
│ 2024 ┆ 62382 ┆ 171        ┆ 0.284         │
│ 2025 ┆ 62382 ┆ 172        ┆ 0.285         │
│ 2026 ┆ 48035 ┆ 174        ┆ 0.286         │
└──────┴───────┴────────────┴───────────────┘
after weekend filter: (450862, 5)


## 4. Fix the split date BEFORE any fitting
Time-based split on unique trading dates (80% train / 20% test)

In [ ]:
SPLIT_FRAC = 0.8
all_dates = df["date"].unique().sort()
CUT = all_dates[int(len(all_dates) * SPLIT_FRAC)]
print("train: dates <", CUT, "| test: dates >=", CUT)

is_train = pl.col("date") < CUT

train: dates < 2022-05-06 | test: dates >= 2022-05-06


## 5. Returns and glitch repair (fixed rule, no fitted parameters)
A *glitch* = a move over 5% that is at least 70% reversed the next day (e.g. BDT 2023-01-04). We keep the raw rate, flag the row, blank the bad rate and forward-fill it.

Caveat: the rule peeks one day ahead, which production can't do. So we keep the raw columns and report test metrics both ways in section 10.

In [ ]:
THR = 0.05          # minimum move to be considered
REVERSAL = 0.3      # next-day return must cancel >= 70% of the move

df = (
    df.with_columns(rate_raw=pl.col("exchange_rate"))
      .with_columns(ret_raw=pl.col("rate_raw").log().diff().over("currency"))
      .with_columns(nxt=pl.col("ret_raw").shift(-1).over("currency"))
)

glitch = (pl.col("ret_raw").abs() > THR) & (
    (pl.col("ret_raw") + pl.col("nxt")).abs() < REVERSAL * pl.col("ret_raw").abs()
)

df = (
    df.with_columns(is_glitch=glitch.fill_null(False))
      .with_columns(
          rate=pl.when(pl.col("is_glitch")).then(None)
                 .otherwise(pl.col("rate_raw"))
                 .fill_null(strategy="forward").over("currency")
      )
      .with_columns(log_ret=pl.col("rate").log().diff().over("currency"))
      .with_columns(
          target=pl.col("log_ret").shift(-1).over("currency"),       # repaired next-day return (train on this)
          target_raw=pl.col("ret_raw").shift(-1).over("currency"),   # raw next-day return (evaluate on this)
          touched=(pl.col("is_glitch")
                   | pl.col("is_glitch").shift(-1).over("currency")).fill_null(False),
      )
      .drop(["nxt", "exchange_rate"])
)

print("glitch rows:", df["is_glitch"].sum(),
      "| in train:", df.filter(is_train)["is_glitch"].sum(),
      "| in test:", df.filter(~is_train)["is_glitch"].sum())

# audit list: look at these before trusting the rule
print(
    df.filter(pl.col("is_glitch"))
      .select(["currency", "date", "rate_raw", "rate", "ret_raw"])
      .sort(pl.col("ret_raw").abs(), descending=True)
      .head(20)
)

glitch rows: 182 | in train: 78 | in test: 104
shape: (20, 5)
┌──────────┬────────────┬────────────┬────────────┬───────────┐
│ currency ┆ date       ┆ rate_raw   ┆ rate       ┆ ret_raw   │
│ ---      ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│ str      ┆ date       ┆ f64        ┆ f64        ┆ f64       │
╞══════════╪════════════╪════════════╪════════════╪═══════════╡
│ BDT      ┆ 2023-01-04 ┆ 2.233      ┆ 108.971    ┆ -3.887736 │
│ FJD      ┆ 2024-06-07 ┆ 0.479209   ┆ 2.458374   ┆ -1.635119 │
│ AOA      ┆ 2025-02-03 ┆ 467.356401 ┆ 948.782666 ┆ -0.708088 │
│ HNL      ┆ 2024-02-15 ┆ 13.544672  ┆ 26.509058  ┆ -0.671493 │
│ DOP      ┆ 2024-01-04 ┆ 36.343314  ┆ 63.251019  ┆ -0.554101 │
│ …        ┆ …          ┆ …          ┆ …          ┆ …         │
│ CDF      ┆ 2016-12-01 ┆ 974.24     ┆ 1215.88    ┆ -0.221566 │
│ PLN      ┆ 2024-01-01 ┆ 5.391954   ┆ 4.343399   ┆ 0.216251  │
│ MNT      ┆ 2016-12-01 ┆ 2115.45    ┆ 2602.95    ┆ -0.207378 │
│ SLL      ┆ 2016-10-19 ┆ 7463.12    ┆ 619

## 6. Coverage per currency (DuckDB, descriptive)

In [ ]:
coverage = duckdb.sql("""
    SELECT currency,
           MIN(date) AS first_date,
           MAX(date) AS last_date,
           COUNT(*)  AS n_rows
    FROM df
    GROUP BY currency
    ORDER BY n_rows DESC
""").pl()

print(coverage.head(15))
print(coverage.tail(10))
print(coverage["n_rows"].describe())

shape: (15, 4)
┌──────────┬────────────┬────────────┬────────┐
│ currency ┆ first_date ┆ last_date  ┆ n_rows │
│ ---      ┆ ---        ┆ ---        ┆ ---    │
│ str      ┆ date       ┆ date       ┆ i64    │
╞══════════╪════════════╪════════════╪════════╡
│ AUD      ┆ 2004-08-30 ┆ 2026-10-07 ┆ 5768   │
│ CAD      ┆ 2004-08-31 ┆ 2026-10-07 ┆ 5766   │
│ JPY      ┆ 2004-08-31 ┆ 2026-10-07 ┆ 5766   │
│ GBP      ┆ 2004-09-03 ┆ 2026-10-07 ┆ 5764   │
│ CHF      ┆ 2004-09-03 ┆ 2026-10-07 ┆ 5764   │
│ …        ┆ …          ┆ …          ┆ …      │
│ ILS      ┆ 2014-11-07 ┆ 2026-10-07 ┆ 3108   │
│ INR      ┆ 2014-11-07 ┆ 2026-10-07 ┆ 3108   │
│ COP      ┆ 2014-11-07 ┆ 2026-10-07 ┆ 3108   │
│ UYU      ┆ 2014-11-07 ┆ 2026-10-07 ┆ 3108   │
│ SGD      ┆ 2014-11-07 ┆ 2026-10-07 ┆ 3108   │
└──────────┴────────────┴────────────┴────────┘
shape: (10, 4)
┌──────────┬────────────┬────────────┬────────┐
│ currency ┆ first_date ┆ last_date  ┆ n_rows │
│ ---      ┆ ---        ┆ ---        ┆ ---    │
│ str     

## 7. Eligibility rules, with data-dependent stats computed on TRAIN only

In [ ]:
EXCLUDE = ["EUR", "XAU", "XAG", "BTC", "XDR",        # base / metals / non-currencies
           "VES", "VEF", "SLL", "MRO"]                 # hyperinflation / redenominated / discontinued
MIN_TRAIN_ROWS = 1000
MIN_TEST_ROWS = 100
MAX_ZERO_SHARE = 0.5

train = df.filter(is_train)
test = df.filter(~is_train)

train_stats = train.group_by("currency").agg(
    n_train=pl.len(),
    zero_share=(pl.col("log_ret") == 0).mean().round(3),
)
test_stats = test.group_by("currency").agg(n_test=pl.len())

latest = df["date"].max()
cutoff = latest - timedelta(days=7)      # availability check, not a fitted statistic

elig = (
    coverage.select(["currency", "last_date"])
    .join(train_stats, on="currency", how="left")
    .join(test_stats, on="currency", how="left")
    .with_columns(pl.col(["n_train", "n_test"]).fill_null(0))
    .with_columns(
        eligible=(
            (pl.col("last_date") >= cutoff)
            & (pl.col("n_train") >= MIN_TRAIN_ROWS)
            & (pl.col("n_test") >= MIN_TEST_ROWS)
            & (pl.col("zero_share") < MAX_ZERO_SHARE)
            & ~pl.col("currency").is_in(EXCLUDE)
        )
    )
)

eligible_currencies = elig.filter(pl.col("eligible"))["currency"].to_list()
print("eligible:", len(eligible_currencies), "of", elig.height)
print(elig.filter(~pl.col("eligible")).sort("n_train").head(40))

eligible: 125 of 176
shape: (40, 6)
┌──────────┬────────────┬─────────┬────────────┬────────┬──────────┐
│ currency ┆ last_date  ┆ n_train ┆ zero_share ┆ n_test ┆ eligible │
│ ---      ┆ ---        ┆ ---     ┆ ---        ┆ ---    ┆ ---      │
│ str      ┆ date       ┆ u32     ┆ f64        ┆ u32    ┆ bool     │
╞══════════╪════════════╪═════════╪════════════╪════════╪══════════╡
│ SLE      ┆ 2026-10-07 ┆ 0       ┆ null       ┆ 1093   ┆ false    │
│ LVL      ┆ 2026-10-07 ┆ 0       ┆ null       ┆ 768    ┆ false    │
│ CVE      ┆ 2026-10-07 ┆ 0       ┆ null       ┆ 768    ┆ false    │
│ BRL      ┆ 2026-10-07 ┆ 0       ┆ null       ┆ 768    ┆ false    │
│ NGN      ┆ 2026-10-07 ┆ 0       ┆ null       ┆ 768    ┆ false    │
│ …        ┆ …          ┆ …       ┆ …          ┆ …      ┆ …        │
│ STN      ┆ 2026-10-07 ┆ 0       ┆ null       ┆ 318    ┆ false    │
│ XCG      ┆ 2026-10-07 ┆ 0       ┆ null       ┆ 318    ┆ false    │
│ MRO      ┆ 2023-12-05 ┆ 0       ┆ null       ┆ 27     ┆ false    

## 8. Target clipping: fit on train, apply anywhere
Per-currency quantile bounds of the (repaired) target. `fit_clip_bounds` sees only train data; `apply_clip` can then be used on train, validation or test with those same numbers. In Phase 3 these bounds get saved with the model in MLflow.

Use the clipped target for **training**. Always **evaluate on the unclipped raw target**.

In [ ]:
Q_LO, Q_HI = 0.005, 0.995

def fit_clip_bounds(train_df: pl.DataFrame) -> pl.DataFrame:
    return (
        train_df.drop_nulls("target")
        .group_by("currency")
        .agg(
            lo=pl.col("target").quantile(Q_LO),
            hi=pl.col("target").quantile(Q_HI),
        )
    )

def apply_clip(frame: pl.DataFrame, bounds: pl.DataFrame) -> pl.DataFrame:
    return (
        frame.join(bounds, on="currency", how="left")
        .with_columns(target_clipped=pl.col("target").clip(pl.col("lo"), pl.col("hi")))
        .drop(["lo", "hi"])
    )

work = df.filter(pl.col("currency").is_in(eligible_currencies))
bounds = fit_clip_bounds(work.filter(is_train))      # TRAIN ONLY
work = apply_clip(work, bounds)

print(bounds.sort("currency").head(10))
changed = work.filter(pl.col("target").is_not_null() & (pl.col("target") != pl.col("target_clipped"))).height
print("target rows changed by clipping:", changed, "of", work.height)

shape: (10, 3)
┌──────────┬───────────┬──────────┐
│ currency ┆ lo        ┆ hi       │
│ ---      ┆ ---       ┆ ---      │
│ str      ┆ f64       ┆ f64      │
╞══════════╪═══════════╪══════════╡
│ AFN      ┆ -0.022967 ┆ 0.023925 │
│ ALL      ┆ -0.014114 ┆ 0.013345 │
│ ANG      ┆ -0.015094 ┆ 0.015007 │
│ ARS      ┆ -0.028717 ┆ 0.042237 │
│ AUD      ┆ -0.018321 ┆ 0.022716 │
│ AWG      ┆ -0.015132 ┆ 0.015581 │
│ AZN      ┆ -0.01917  ┆ 0.019629 │
│ BAM      ┆ -0.01185  ┆ 0.011634 │
│ BBD      ┆ -0.021196 ┆ 0.020442 │
│ BDT      ┆ -0.017237 ┆ 0.018002 │
└──────────┴───────────┴──────────┘
target rows changed by clipping: 3998 of 400065


## 9. Look at a few currencies

In [ ]:
SEL = ["USD", "JPY", "LKR"]
sub = work.filter(pl.col("currency").is_in(SEL)).to_pandas()

fig = px.line(sub, x="date", y="rate", facet_row="currency", title="EUR -> currency (repaired rate)")
fig.update_yaxes(matches=None)
fig.add_vline(x=str(CUT), line_dash="dash")
fig.show()

fig = px.line(sub, x="date", y="log_ret", facet_row="currency", title="Daily log return")
fig.update_yaxes(matches=None)
fig.add_vline(x=str(CUT), line_dash="dash")
fig.show()

## 10. Baselines on the test window
Naive predictors for the next-day return: **zero** and **persistence** (yesterday's return). Reported two ways: all rows against the *raw* target, and excluding rows touched by a repaired glitch. If the two agree, the repair doesn't flatter the result.

In [ ]:
PROTO = ["USD", "GBP", "JPY", "CHF", "AUD", "CAD", "LKR"]

t = (
    work.filter(~is_train & pl.col("currency").is_in(PROTO))
        .drop_nulls(["log_ret", "target_raw"])
        .with_columns(
            err_zero=pl.col("target_raw").abs(),
            err_persist=(pl.col("target_raw") - pl.col("log_ret")).abs(),
        )
)

def summarize(frame: pl.DataFrame) -> pl.DataFrame:
    return (
        frame.group_by("currency")
        .agg(
            mae_zero=pl.col("err_zero").mean(),
            mae_persist=pl.col("err_persist").mean(),
            n=pl.len(),
        )
        .sort("currency")
    )

print("ALL ROWS (raw target)")
print(summarize(t))
print("EXCLUDING ROWS TOUCHED BY A REPAIRED GLITCH")
print(summarize(t.filter(~pl.col("touched"))))
print("overall MAE zero | all rows:", t["err_zero"].mean(),
      "| excluding touched:", t.filter(~pl.col("touched"))["err_zero"].mean())

# the 7 prototype currencies may have no glitches at all; check every eligible currency too
allt = (
    work.filter(~is_train)
        .drop_nulls(["log_ret", "target_raw"])
        .with_columns(err_zero=pl.col("target_raw").abs())
)
print("ALL ELIGIBLE currencies, zero-baseline MAE | all rows:", allt["err_zero"].mean(),
      "| excluding touched:", allt.filter(~pl.col("touched"))["err_zero"].mean(),
      "| touched rows:", allt["touched"].sum(), "of", allt.height)

ALL ROWS (raw target)
shape: (7, 4)
┌──────────┬──────────┬─────────────┬──────┐
│ currency ┆ mae_zero ┆ mae_persist ┆ n    │
│ ---      ┆ ---      ┆ ---         ┆ ---  │
│ str      ┆ f64      ┆ f64         ┆ u32  │
╞══════════╪══════════╪═════════════╪══════╡
│ AUD      ┆ 0.003451 ┆ 0.005125    ┆ 1153 │
│ CAD      ┆ 0.002832 ┆ 0.004223    ┆ 1153 │
│ CHF      ┆ 0.002455 ┆ 0.003544    ┆ 1153 │
│ GBP      ┆ 0.002277 ┆ 0.003248    ┆ 1153 │
│ JPY      ┆ 0.004211 ┆ 0.00615     ┆ 1153 │
│ LKR      ┆ 0.004416 ┆ 0.006407    ┆ 1130 │
│ USD      ┆ 0.003466 ┆ 0.005036    ┆ 1153 │
└──────────┴──────────┴─────────────┴──────┘
EXCLUDING ROWS TOUCHED BY A REPAIRED GLITCH
shape: (7, 4)
┌──────────┬──────────┬─────────────┬──────┐
│ currency ┆ mae_zero ┆ mae_persist ┆ n    │
│ ---      ┆ ---      ┆ ---         ┆ ---  │
│ str      ┆ f64      ┆ f64         ┆ u32  │
╞══════════╪══════════╪═════════════╪══════╡
│ AUD      ┆ 0.003451 ┆ 0.005125    ┆ 1153 │
│ CAD      ┆ 0.002832 ┆ 0.004223    ┆ 1153 │
│ CHF 

## 11. Conclusions (fill in after running)
- Key `(currency, date)`; weekends dropped; target = next-trading-day log return of EUR->X
- Split date (hard-code it): ____
- Glitch rule: THR=0.05, REVERSAL=0.3; repaired rows: ____ (train ____ / test ____)
- Eligible currencies: ____ (rules in section 7, train-only stats)
- Clip: per-currency quantiles 0.5% / 99.5% from train; bounds saved with the model
- Baseline MAE to beat (zero predictor): ____ (all rows) / ____ (excluding touched)
- Prototype currencies: USD, GBP, JPY, CHF, AUD, CAD, LKR